In [2]:
# CELL 1: IMPORTS & INSTALL CHECK

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, os, joblib, json, time
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, average_precision_score

# Check FastAPI
try:
    import fastapi, uvicorn, pydantic
    print(f' FastAPI {fastapi.__version__} | Pydantic {pydantic.__version__}')
    FASTAPI_OK = True
except ImportError:
    FASTAPI_OK = False
    print('  FastAPI not installed. Run: pip install fastapi uvicorn httpx')

os.makedirs('../results', exist_ok=True)
os.makedirs('../deploy',  exist_ok=True)

print('\n' + '='*60)
print('   NOTEBOOK 07: FASTAPI REST DEPLOYMENT')
print('='*60)
print('Goal: wrap the trained XGBoost model in a production REST API')

  FastAPI not installed. Run: pip install fastapi uvicorn httpx

   NOTEBOOK 07: FASTAPI REST DEPLOYMENT
Goal: wrap the trained XGBoost model in a production REST API


In [3]:
# CELL 2: LOAD & BUNDLE MODEL ARTIFACTS
# Bundle model + scaler + metadata into one object
# → single file to deploy, easy version management

print('=== Packaging Model Artifact ===')

# Load components
model         = joblib.load('../results/model_xgb.pkl')
scaler        = joblib.load('../results/scaler.pkl')
feature_names = pd.read_csv('../data/feature_names.csv')['feature'].tolist()

# Quick validation
X_test = np.load('../data/X_test.npy')
y_test = np.load('../data/y_test.npy')
test_probs = model.predict_proba(X_test)[:, 1]
test_auc   = roc_auc_score(y_test, test_probs)
test_ap    = average_precision_score(y_test, test_probs)

# Bundle into single artifact dict
artifact = {
    'model'         : model,
    'scaler'        : scaler,
    'feature_names' : feature_names,
    'threshold'     : 0.30,          # business cost-optimised in NB03
    'metadata': {
        'model_type'    : 'XGBoost',
        'n_estimators'  : model.n_estimators,
        'n_features'    : len(feature_names),
        'train_auc_roc' : round(test_auc, 4),
        'train_avg_prec': round(test_ap, 4),
        'threshold'     : 0.30,
        'version'       : '1.0.0',
        'trained_on'    : 'Kaggle ULB Credit Card Fraud (284,807 tx)',
    }
}

# Save bundled artifact
joblib.dump(artifact, '../deploy/fraud_model_artifact.pkl')

print(f'Model loaded        : XGBoost ({model.n_estimators} trees)')
print(f'Features            : {len(feature_names)}')
print(f'Test AUC-ROC        : {test_auc:.4f}')
print(f'Test Avg Precision  : {test_ap:.4f}')
print(f'Decision threshold  : 0.30')
print(f'\n Bundled artifact → deploy/fraud_model_artifact.pkl')

=== Packaging Model Artifact ===
Model loaded        : XGBoost (500 trees)
Features            : 32
Test AUC-ROC        : 0.9782
Test Avg Precision  : 0.8328
Decision threshold  : 0.30

 Bundled artifact → deploy/fraud_model_artifact.pkl


In [5]:
api_code = """
# Credit Card Fraud Detection — FastAPI REST Service

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List
import numpy as np
import pandas as pd
import joblib
import time
import os

# Load Model Artifact
ARTIFACT_PATH = os.getenv("MODEL_PATH", "../deploy/fraud_model_artifact.pkl")

artifact      = joblib.load(ARTIFACT_PATH)
MODEL         = artifact["model"]
SCALER        = artifact["scaler"]
FEATURE_NAMES = artifact["feature_names"]
THRESHOLD     = artifact["threshold"]
METADATA      = artifact["metadata"]
N_FEATURES    = len(FEATURE_NAMES)

# FastAPI App
app = FastAPI(
    title="Credit Card Fraud Detection API",
    description="Real-time fraud scoring using XGBoost",
    version=METADATA.get("version", "1.0.0")
)

# Input Schema
class TransactionFeatures(BaseModel):
    Time: float = Field(..., ge=0)
    Amount: float = Field(..., ge=0)

    V1: float
    V2: float
    V3: float
    V4: float
    V5: float
    V6: float
    V7: float
    V8: float
    V9: float
    V10: float
    V11: float
    V12: float
    V13: float
    V14: float
    V15: float
    V16: float
    V17: float
    V18: float
    V19: float
    V20: float
    V21: float
    V22: float
    V23: float
    V24: float
    V25: float
    V26: float
    V27: float
    V28: float


class PredictionResponse(BaseModel):
    fraud_probability: float
    fraud_predicted: bool
    risk_level: str
    threshold_used: float
    inference_ms: float
    model_version: str


class BatchRequest(BaseModel):
    transactions: List[TransactionFeatures]


class BatchResponse(BaseModel):
    results: List[PredictionResponse]
    total_transactions: int
    fraud_flagged: int
    total_ms: float
    avg_ms_per_tx: float


# Risk Level Function
def risk_level(prob):
    if prob < 0.30:
        return "LOW"
    elif prob < 0.70:
        return "MEDIUM"
    else:
        return "HIGH"


# Preprocess Function
def preprocess(tx: TransactionFeatures):
    hour = (tx.Time / 3600) % 24
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)

    log_amount = np.log1p(tx.Amount)

    v_features = [getattr(tx, f"V{i}") for i in range(1, 29)]

    raw_vec = np.array(
        v_features + [tx.Amount, log_amount, hour_sin, hour_cos]
    ).reshape(1, -1)

    scaled = SCALER.transform(raw_vec)

    return scaled


# Health Check
@app.get("/")
def health():
    return {
        "status": "healthy",
        "service": "Fraud Detection API",
        "version": METADATA.get("version")
    }


# Model Info
@app.get("/model/info")
def model_info():
    return {
        "model_metadata": METADATA,
        "n_features": N_FEATURES,
        "feature_names": FEATURE_NAMES,
        "decision_threshold": THRESHOLD
    }


# Single Prediction
@app.post("/predict", response_model=PredictionResponse)
def predict(transaction: TransactionFeatures):

    t0 = time.time()

    try:
        X = preprocess(transaction)
        prob = float(MODEL.predict_proba(X)[0, 1])
    except Exception as e:
        raise HTTPException(status_code=422, detail=str(e))

    elapsed = (time.time() - t0) * 1000

    return PredictionResponse(
        fraud_probability=round(prob, 6),
        fraud_predicted=prob >= THRESHOLD,
        risk_level=risk_level(prob),
        threshold_used=THRESHOLD,
        inference_ms=round(elapsed, 3),
        model_version=METADATA.get("version", "1.0.0")
    )


# Batch Prediction
@app.post("/predict/batch", response_model=BatchResponse)
def predict_batch(request: BatchRequest):

    t0 = time.time()

    results = []

    for tx in request.transactions:

        t_tx = time.time()

        X = preprocess(tx)
        prob = float(MODEL.predict_proba(X)[0, 1])

        ms = (time.time() - t_tx) * 1000

        results.append(
            PredictionResponse(
                fraud_probability=round(prob, 6),
                fraud_predicted=prob >= THRESHOLD,
                risk_level=risk_level(prob),
                threshold_used=THRESHOLD,
                inference_ms=round(ms, 3),
                model_version=METADATA.get("version", "1.0.0")
            )
        )

    total_ms = (time.time() - t0) * 1000
    flagged = sum(r.fraud_predicted for r in results)

    return BatchResponse(
        results=results,
        total_transactions=len(results),
        fraud_flagged=flagged,
        total_ms=round(total_ms, 2),
        avg_ms_per_tx=round(total_ms / len(results), 3)
    )
"""

# Write file with UTF-8 encoding (fixes the error)
with open("../deploy/app.py", "w", encoding="utf-8") as f:
    f.write(api_code)

print("FastAPI app written successfully to deploy/app.py")

print("\nTo run the API:")
print("cd deploy")
print("uvicorn app:app --host 0.0.0.0 --port 8000 --reload")

print("\nOpen API docs:")
print("http://localhost:8000/docs")

FastAPI app written successfully to deploy/app.py

To run the API:
cd deploy
uvicorn app:app --host 0.0.0.0 --port 8000 --reload

Open API docs:
http://localhost:8000/docs


In [6]:
# CELL 4: GENERATE DOCKERFILE

dockerfile = """# Credit Card Fraud Detection Service
FROM python:3.10-slim

WORKDIR /app

# Install dependencies
COPY requirements_deploy.txt .
RUN pip install --no-cache-dir -r requirements_deploy.txt

# Copy model artifact and API code
COPY fraud_model_artifact.pkl .
COPY app.py .

# Environment
ENV MODEL_PATH=/app/fraud_model_artifact.pkl
ENV PORT=8000

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:8000/ || exit 1

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
"""

requirements_deploy = """fastapi>=0.104.0
uvicorn[standard]>=0.24.0
pydantic>=2.0.0
numpy>=1.24.0
pandas>=2.0.0
xgboost>=2.0.0
scikit-learn>=1.3.0
joblib>=1.3.0
httpx>=0.25.0
"""

with open('../deploy/Dockerfile', 'w') as f:
    f.write(dockerfile)
with open('../deploy/requirements_deploy.txt', 'w') as f:
    f.write(requirements_deploy)

print('  Dockerfile             → deploy/Dockerfile')
print('  requirements_deploy.txt → deploy/requirements_deploy.txt')
print('\nTo build & run with Docker:')
print('  docker build -t fraud-api ./deploy')
print('  docker run -p 8000:8000 fraud-api')

  Dockerfile             → deploy/Dockerfile
  requirements_deploy.txt → deploy/requirements_deploy.txt

To build & run with Docker:
  docker build -t fraud-api ./deploy
  docker run -p 8000:8000 fraud-api
